# 조회 쿼리 모음 — 모두몰 데이터
- 작성자: 박소영
- 실행 환경: Google Colab / Jupyter + DuckDB
- 데이터: 수업 [C2] `customers`, [C3] `orders` 테이블

`SELECT`·`FROM`·`WHERE`와 비교·논리 연산자(`=`, `!=`, `>=`, `<`, `AND`, `OR`, `NOT`, `IN`, `BETWEEN`), `IS NULL`을 활용해 질문 5개에 답합니다.

## 0. 준비 — DuckDB 연결 및 테이블 만들기

In [1]:
%pip install -q duckdb pandas

import duckdb
con = duckdb.connect()  # 메모리 안에 임시 DB 생성

def run(sql):
    """SQL을 실행하고 결과를 표(DataFrame)로 돌려줍니다."""
    return con.sql(sql).df()

In [2]:
# 수업 [C2], [C3]의 테이블 생성 + 데이터 입력
con.execute("""
CREATE OR REPLACE TABLE customers (
  customer_id STRING, name STRING, country STRING, signup_date DATE, grade STRING
);
INSERT INTO customers VALUES
  ('C001', '김민준', 'Korea', '2023-01-05', 'Gold'),
  ('C002', '이서연', 'Korea', '2023-02-11', 'Silver'),
  ('C003', '박도윤', 'Japan', '2023-02-20', 'Bronze'),
  ('C004', '최지우', 'USA', '2023-03-03', 'Gold'),
  ('C005', '정하준', 'Korea', '2023-03-15', 'Silver'),
  ('C006', '강서윤', 'Korea', '2023-04-01', 'Bronze'),
  ('C007', '조은우', 'Japan', '2023-04-18', 'Silver'),
  ('C008', '윤지호', 'USA', '2023-05-09', 'Gold'),
  ('C009', '임하은', 'Korea', '2023-05-22', 'Bronze'),
  ('C010', '한예준', 'Korea', '2023-06-02', 'Silver'),
  ('C011', '오시우', NULL, '2023-06-19', 'Bronze'),
  ('C012', '신아린', 'Japan', '2023-07-07', 'Silver'),
  ('C013', '권준서', 'Korea', '2023-07-25', 'Gold'),
  ('C014', '황지안', 'USA', '2023-08-10', NULL),
  ('C015', '안수아', 'Korea', '2023-08-28', 'Bronze');
CREATE OR REPLACE TABLE orders (
  order_id STRING, customer_id STRING, order_date DATE, status STRING, amount DECIMAL(12,2)
);
INSERT INTO orders VALUES
  ('O0001', 'C001', '2023-09-02', 'Paid', 125000),
  ('O0002', 'C002', '2023-09-05', 'Shipped', 89000),
  ('O0003', 'C001', '2023-09-11', 'Returned', 45000),
  ('O0004', 'C003', '2023-09-15', 'Paid', 230000),
  ('O0005', 'C004', '2023-09-20', 'Cancelled', NULL),
  ('O0006', 'C005', '2023-09-25', 'Shipped', 67000),
  ('O0007', 'C002', '2023-10-01', 'Paid', 158000),
  ('O0008', 'C006', '2023-10-04', 'Placed', 32000),
  ('O0009', 'C007', '2023-10-12', 'Shipped', 410000),
  ('O0010', 'C008', '2023-10-19', 'Paid', 99000),
  ('O0011', 'C001', '2023-10-23', 'Paid', 76000),
  ('O0012', 'C009', '2023-10-28', 'Cancelled', NULL),
  ('O0013', 'C010', '2023-11-02', 'Shipped', 142000),
  ('O0014', 'C004', '2023-11-08', 'Paid', 88000),
  ('O0015', 'C011', '2023-11-13', 'Placed', 53000),
  ('O0016', 'C012', '2023-11-19', 'Shipped', 175000),
  ('O0017', 'C002', '2023-11-24', 'Returned', 61000),
  ('O0018', 'C013', '2023-11-29', 'Paid', 320000),
  ('O0019', 'C005', '2023-12-03', 'Paid', 47000),
  ('O0020', 'C008', '2023-12-09', 'Shipped', 215000),
  ('O0021', 'C014', '2023-12-14', 'Placed', 38000),
  ('O0022', 'C001', '2023-12-20', 'Paid', 134000),
  ('O0023', 'C015', '2023-12-25', 'Shipped', 92000),
  ('O0024', 'C007', '2024-01-03', 'Paid', 268000),
  ('O0025', 'C010', '2024-01-09', 'Cancelled', NULL),
  ('O0026', 'C003', '2024-01-15', 'Paid', 119000),
  ('O0027', 'C013', '2024-01-22', 'Shipped', 405000),
  ('O0028', 'C006', '2024-01-28', 'Paid', 58000),
  ('O0029', 'C004', '2024-02-04', 'Returned', 73000),
  ('O0030', 'C012', '2024-02-11', 'Paid', 187000);
""")
print("테이블 생성 완료!")

테이블 생성 완료!


### 스키마 확인

In [3]:
run("DESCRIBE customers")

,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,YES,None,None,None
1,name,VARCHAR,YES,None,None,None
2,country,VARCHAR,YES,None,None,None
3,signup_date,DATE,YES,None,None,None
4,grade,VARCHAR,YES,None,None,None


In [4]:
run("DESCRIBE orders")

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,order_date,DATE,YES,None,None,None
3,status,VARCHAR,YES,None,None,None
4,amount,"DECIMAL(12,2)",YES,None,None,None


## Q1. Gold 등급인 한국 고객은 누구인가?
사용한 패턴: `=`, `AND`

In [5]:
run("""
SELECT name, country, grade
FROM customers
WHERE grade = 'Gold'
  AND country = 'Korea'
""")

,name,country,grade
0,김민준,Korea,Gold
1,권준서,Korea,Gold


**발견한 점:** 한국 고객 8명 중 Gold 등급은 김민준·권준서 2명(25%)뿐이고, 전체 Gold 4명 중 나머지 2명은 미국 고객이다.

## Q2. 해외(일본·미국) 고객 중 Gold 등급이 아닌 고객은 누구인가?
사용한 패턴: `IN`, `NOT IN`, `AND`

In [6]:
run("""
SELECT name, country, grade
FROM customers
WHERE country IN ('Japan', 'USA')
  AND grade NOT IN ('Gold')
ORDER BY country
""")

,name,country,grade
0,박도윤,Japan,Bronze
1,조은우,Japan,Silver
2,신아린,Japan,Silver


**발견한 점:** 일본 고객 3명만 나오고 미국 고객은 0명이다. 미국 고객 중 등급이 NULL인 황지안은 Gold가 아닌데도 빠졌는데, NULL과의 비교 결과가 UNKNOWN이라 `NOT IN`에 걸리지 않기 때문이다.

## Q3. 국가나 등급 정보가 비어 있는 고객은 누구인가? (데이터 품질 점검)
사용한 패턴: `SELECT *`, `IS NULL`, `OR`

In [7]:
run("""
SELECT *
FROM customers
WHERE country IS NULL
   OR grade IS NULL
""")

,customer_id,name,country,signup_date,grade
0,C011,오시우,NaN,2023-06-19,Bronze
1,C014,황지안,USA,2023-08-10,NaN


**발견한 점:** 15명 중 2명(오시우: 국가 없음, 황지안: 등급 없음)의 정보가 비어 있어서, 국가별·등급별 분석을 할 때 이 고객들이 빠지지 않게 따로 처리해야 한다.

## Q4. 2023년 4분기(10~12월) 10만 원 이상 주문과 부가세(10%) 포함 금액은?
사용한 패턴: `BETWEEN`, `>=`, `AND`, 계산(`amount * 1.1`)

In [8]:
run("""
SELECT order_id,
       order_date,
       status,
       amount,
       amount * 1.1 AS amount_with_vat
FROM orders
WHERE order_date BETWEEN '2023-10-01' AND '2023-12-31'
  AND amount >= 100000
ORDER BY amount DESC
""")

,order_id,order_date,status,amount,amount_with_vat
0,O0009,2023-10-12,Shipped,410000.0,451000.0
1,O0018,2023-11-29,Paid,320000.0,352000.0
2,O0020,2023-12-09,Shipped,215000.0,236500.0
3,O0016,2023-11-19,Shipped,175000.0,192500.0
4,O0007,2023-10-01,Paid,158000.0,173800.0
5,O0013,2023-11-02,Shipped,142000.0,156200.0
6,O0022,2023-12-20,Paid,134000.0,147400.0


**발견한 점:** 4분기 주문 17건 중 7건이 10만 원 이상 고액 주문이고, 모두 Paid 또는 Shipped 상태라 취소·반품된 고액 주문은 없었다.

## Q5. 취소되지 않은 주문 중 아주 작은(5만 원 미만) 주문과 아주 큰(30만 원 이상) 주문은?
사용한 패턴: `!=`, `<`, `>=`, `AND`, `OR`, 괄호

In [9]:
run("""
SELECT order_id, customer_id, order_date, status, amount
FROM orders
WHERE status != 'Cancelled'
  AND (amount < 50000 OR amount >= 300000)
ORDER BY amount
""")

,order_id,customer_id,order_date,status,amount
0,O0008,C006,2023-10-04,Placed,32000.0
1,O0021,C014,2023-12-14,Placed,38000.0
2,O0003,C001,2023-09-11,Returned,45000.0
3,O0019,C005,2023-12-03,Paid,47000.0
4,O0018,C013,2023-11-29,Paid,320000.0
5,O0027,C013,2024-01-22,Shipped,405000.0
6,O0009,C007,2023-10-12,Shipped,410000.0


**발견한 점:** 5만 원 미만 주문 4건 중 3건은 아직 결제 전(Placed)이거나 반품(Returned)인 반면, 30만 원 이상 주문 3건은 모두 Paid·Shipped이며 그중 2건이 고객 C013(권준서) 한 사람의 주문이다.

### 참고: Q5에서 괄호가 필요한 이유
`AND`가 `OR`보다 먼저 계산되기 때문에, 괄호가 없으면
`(status != 'Cancelled' AND amount < 50000) OR amount >= 300000`으로 해석됩니다.
→ 30만 원 이상 주문은 **취소 여부와 상관없이** 포함되어 의도와 달라집니다.

## 조건 체크
| 조건 | 사용한 곳 |
|---|---|
| 비교 연산자 `=`, `!=`, `>=`, `<` | Q1, Q5, Q4·Q5, Q5 |
| 논리 연산자 `AND`, `OR`, `NOT` | Q1~Q5, Q3·Q5, Q2 |
| `IN`, `BETWEEN` | Q2, Q4 |
| `IS NULL` | Q3 |